# Hodnocení AI modelů: Hodnocení pomocí kódu, lidmi a modely

V tomto notebooku se ponoříme do trojice široce používaných technik pro hodnocení účinnosti AI modelů, jako je Claude v3:

1. Hodnocení pomocí kódu
2. Hodnocení lidmi
3. Hodnocení pomocí modelů

Každý přístup ilustrujeme pomocí příkladů a prozkoumáme jejich příslušné výhody a omezení při posuzování výkonu AI.

## Příklad hodnocení založeného na kódu: Analýza sentimentu

V tomto příkladu vyhodnotíme schopnost Claude klasifikovat sentiment filmových recenzí jako pozitivní nebo negativní. Můžeme použít kód k ověření, zda výstup modelu odpovídá očekávanému sentimentu.

In [ ]:
```python
# Uložte název modelu a AWS region pro pozdější použití
MODEL_NAME = "anthropic.claude-3-haiku-20240307-v1:0"
AWS_REGION = "us-west-2"

%store MODEL_NAME
%store AWS_REGION
```

In [ ]:
# Nainstalujte balíček Anthropic
%pip install anthropic --quiet

In [ ]:
```python
# Importujte třídu AnthropicBedrock a vytvořte instanci klienta
from anthropic import AnthropicBedrock

client = AnthropicBedrock(aws_region=AWS_REGION)
```

In [ ]:
```python
# Funkce pro vytvoření vstupního promptu pro analýzu sentimentu
def build_input_prompt(review):
    user_content = f"""Classify the sentiment of the following movie review as either 'positive' or 'negative' provide only one of those two choices:
    <review>{review}</review>"""
    return [{"role": "user", "content": user_content}]

# Definovat hodnotící data
eval = [
    {
        "review": "This movie was amazing! The acting was superb and the plot kept me engaged from start to finish.",
        "golden_answer": "positive"
    },
    {
        "review": "I was thoroughly disappointed by this film. The pacing was slow and the characters were one-dimensional.",
        "golden_answer": "negative"
    }
]

# Funkce pro získání dokončení z modelu
def get_completion(messages):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=messages
    )
    return message.content[0].text

# Získat dokončení pro každý vstup
outputs = [get_completion(build_input_prompt(item["review"])) for item in eval]

# Vytisknout výstupy a zlaté odpovědi
for output, question in zip(outputs, eval):
    print(f"Review: {question['review']}\nGolden Answer: {question['golden_answer']}\nOutput: {output}\n")

# Funkce pro hodnocení dokončení
def grade_completion(output, golden_answer):
    return output.lower() == golden_answer.lower()

# Ohodnotit dokončení a vytisknout přesnost
grades = [grade_completion(output, item["golden_answer"]) for output, item in zip(outputs, eval)]
print(f"Accuracy: {sum(grades) / len(grades) * 100}%")
```

## Příklad hodnocení člověkem: Skórování esejí

Některé úkoly, jako je skórování esejí, je obtížné hodnotit pouze pomocí kódu. V tomto případě můžeme poskytnout pokyny pro lidské hodnotitele, aby posoudili výstup modelu.

In [ ]:
```python
# Funkce pro vytvoření vstupního promptu pro generování eseje
def build_input_prompt(topic):
    user_content = f"""Write a short essay discussing the following topic:
    <topic>{topic}</topic>"""
    return [{"role": "user", "content": user_content}]

# Definování hodnotících dat
eval = [
    {
        "topic": "The importance of education in personal development and societal progress",
        "golden_answer": "Vysoce hodnocená esej by měla mít jasnou tezi, dobře strukturované odstavce a přesvědčivé příklady diskutující, jak vzdělání přispívá k individuálnímu růstu a širšímu společenskému pokroku."
    }
]

# Získání dokončení pro každý vstup
outputs = [get_completion(build_input_prompt(item["topic"])) for item in eval]

# Tisk výstupů a zlatých odpovědí
for output, item in zip(outputs, eval):
    print(f"Topic: {item['topic']}\n\nGrading Rubric:\n {item['golden_answer']}\n\nModel Output:\n{output}\n")
```

## Příklady hodnocení založeného na modelu

Můžeme použít Claude k hodnocení jeho vlastních výstupů tím, že poskytneme odpověď modelu a hodnotící kritéria. To nám umožňuje automatizovat hodnocení úkolů, které by obvykle vyžadovaly lidský úsudek.

### Example 1: Shrnutí

V tomto příkladu použijeme Claude k posouzení kvality shrnutí, které vygeneroval. To může být užitečné, když potřebujete zhodnotit schopnost modelu zachytit klíčové informace z delšího textu stručně a přesně. Poskytnutím rubriky, která vymezuje základní body, které by měly být pokryty, můžeme automatizovat proces hodnocení a rychle posoudit výkon modelu při úkolech shrnutí.

In [ ]:
```python
# Funkce pro vytvoření vstupního promptu pro shrnutí
def build_input_prompt(text):
    user_content = f"""Please summarize the main points of the following text:
    <text>{text}</text>"""
    return [{"role": "user", "content": user_content}]

# Funkce pro vytvoření promptu pro hodnocení kvality shrnutí
def build_grader_prompt(output, rubric):
    user_content = f"""Assess the quality of the following summary based on this rubric:
    <rubric>{rubric}</rubric>
    <summary>{output}</summary>
    Provide a score from 1-5, where 1 is poor and 5 is excellent."""
    return [{"role": "user", "content": user_content}]

# Definovat hodnotící data
eval = [
    {
        "text": "The Magna Carta, signed in 1215, was a pivotal document in English history. It limited the powers of the monarchy and established the principle that everyone, including the king, was subject to the law. This laid the foundation for constitutional governance and the rule of law in England and influenced legal systems worldwide.",
        "golden_answer": "A high-quality summary should concisely capture the key points: 1) The Magna Carta's significance in English history, 2) Its role in limiting monarchical power, 3) Establishing the principle of rule of law, and 4) Its influence on legal systems around the world."
    }
]

# Získat dokončení pro každý vstup
outputs = [get_completion(build_input_prompt(item["text"])) for item in eval]

# Ohodnotit dokončení
grades = [get_completion(build_grader_prompt(output, item["golden_answer"])) for output, item in zip(outputs, eval)]

# Vytisknout skóre kvality shrnutí
print(f"Summary quality score: {grades[0]}")
```

### Příklad 2: Ověřování faktů

V tomto příkladu použijeme Claude k ověření tvrzení a poté posoudíme přesnost jeho ověřování faktů. To může být užitečné, když potřebujete vyhodnotit schopnost modelu rozlišovat mezi přesnými a nepřesnými informacemi. Poskytnutím rubriky, která uvádí klíčové body, které by měly být pokryty ve správném ověření faktů, můžeme automatizovat proces hodnocení a rychle posoudit výkon modelu v úkolech ověřování faktů.

In [ ]:
```python
# Funkce pro vytvoření vstupního promptu pro ověřování faktů
def build_input_prompt(claim):
    user_content = f"""Určete, zda je následující tvrzení pravdivé nebo nepravdivé:
    <claim>{claim}</claim>"""
    return [{"role": "user", "content": user_content}]

# Funkce pro vytvoření promptu pro hodnocení přesnosti ověření faktů
def build_grader_prompt(output, rubric):
    user_content = f"""Na základě následujícího rubriky posuďte, zda je ověření faktů správné:
    <rubric>{rubric}</rubric>
    <fact-check>{output}</fact-check>"""
    return [{"role": "user", "content": user_content}]

# Definujte hodnotící data
eval = [
    {
        "claim": "The Great Wall of China is visible from space.",
        "golden_answer": "Správné ověření faktů by mělo uvést, že toto tvrzení je nepravdivé. I když je Velká čínská zeď impozantní stavbou, není viditelná z vesmíru pouhým okem."
    }
]

# Získání dokončení pro každý vstup
outputs = [get_completion(build_input_prompt(item["claim"])) for item in eval]

grades = []
for output, item in zip(outputs, eval):
    # Vytiskněte tvrzení, ověření faktů a rubriku
    print(f"Claim: {item['claim']}\n")
    print(f"Fact-check: {output}]\n")
    print(f"Rubric: {item['golden_answer']}\n")
    
    # Ohodnoťte ověření faktů
    grader_prompt = build_grader_prompt(output, item["golden_answer"])
    grade = get_completion(grader_prompt)
    grades.append("correct" in grade.lower())

# Vytiskněte přesnost ověřování faktů
accuracy = sum(grades) / len(grades)
print(f"Fact-checking accuracy: {accuracy * 100}%")
```

### Example 3: Analýza tónu

V tomto příkladu použijeme Claude k analýze tónu daného textu a poté posoudíme přesnost jeho analýzy. To může být užitečné, když potřebujete vyhodnotit schopnost modelu identifikovat a interpretovat emocionální obsah a postoje vyjádřené v textu. Poskytnutím rubriky, která popisuje klíčové aspekty tónu, které by měly být identifikovány, můžeme automatizovat proces hodnocení a rychle posoudit výkon modelu v úlohách analýzy tónu.

In [ ]:
```python
# Funkce pro vytvoření vstupního promptu pro analýzu tónu
def build_input_prompt(text):
    user_content = f"""Analyze the tone of the following text:
    <text>{text}</text>"""
    return [{"role": "user", "content": user_content}]

# Funkce pro vytvoření promptu pro hodnocení přesnosti analýzy tónu
def build_grader_prompt(output, rubric):
    user_content = f"""Assess the accuracy of the following tone analysis based on this rubric:
    <rubric>{rubric}</rubric>
    <tone-analysis>{output}</tone-analysis>"""
    return [{"role": "user", "content": user_content}]

# Definovat hodnotící data
eval = [
    {
        "text": "I can't believe they canceled the event at the last minute. This is completely unacceptable and unprofessional!",
        "golden_answer": "The tone analysis should identify the text as expressing frustration, anger, and disappointment. Key words like 'can't believe', 'last minute', 'unacceptable', and 'unprofessional' indicate strong negative emotions."
    }
]

# Získat dokončení pro každý vstup
outputs = [get_completion(build_input_prompt(item["text"])) for item in eval]

# Ohodnotit dokončení
grades = [get_completion(build_grader_prompt(output, item["golden_answer"])) for output, item in zip(outputs, eval)]

# Vytisknout kvalitu analýzy tónu
print(f"Tone analysis quality: {grades[0]}")
```

Tyto příklady ukazují, jak lze použít hodnocení založené na kódu, lidské hodnocení a hodnocení založené na modelu k vyhodnocení AI modelů, jako je Claude, při různých úlohách. Volba metody hodnocení závisí na povaze úlohy a dostupných zdrojích. Hodnocení založené na modelu nabízí slibný přístup k automatizaci hodnocení složitých úloh, které by jinak vyžadovaly lidský úsudek.